# WP3 — Personal Dataset & YOLO Detection Pipeline

End-to-end pipeline from raw smartphone photos to labeled Re-ID crops.

**Contents:**
1. Setup & Configuration
2. YOLO Person Detection Demo
3. Batch Processing — Raw Photos → Crops
4. Manual Annotation Workflow
5. Dataset Splitting (Part 1 / Part 2)
6. PersonalDataset Class Integration
7. Exploratory Data Analysis (EDA)

## 1. Setup & Configuration

In [1]:
import os
import sys
import json
import glob
import yaml

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

/var/folders/5l/rft89vf91_sc94km3w2pwq400000gn/T/ipykernel_25963/1019484203.py:9: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
sys.path.append(os.getcwd())

In [3]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)["reid_project"]

RAW_PHOTOS_DIR = config["personal_data"]["raw_photos_dir"]
CROPS_DIR = config["personal_data"]["detected_crops_dir"]
YOLO_MODEL = config["personal_data"]["yolo_model"]
DETECTION_CONF = config["personal_data"]["detection_threshold"]

print(f"Raw photos directory : {RAW_PHOTOS_DIR}")
print(f"Crops output directory: {CROPS_DIR}")
print(f"YOLO model           : {YOLO_MODEL}")
print(f"Detection confidence : {DETECTION_CONF}")

Raw photos directory : data/personal/raw
Crops output directory: data/personal/crops
YOLO model           : yolo11n.pt
Detection confidence : 0.5


In [4]:
# Create directories if they don't exist
os.makedirs(RAW_PHOTOS_DIR, exist_ok=True)
os.makedirs(CROPS_DIR, exist_ok=True)
print("Directories ready.")

Directories ready.


## 2. YOLO Person Detection Demo

Initialize the YOLO-based person detector and run it on a single image
to verify the pipeline works correctly.

In [5]:
from src.detection.yolo_pipeline import YOLOPersonDetector

detector = YOLOPersonDetector(
    model_name=YOLO_MODEL,
    confidence=DETECTION_CONF,
    iou_threshold=0.45,
    min_height=50,
    min_width=25,
    min_aspect_ratio=1.0,
    max_aspect_ratio=6.0,
    padding_ratio=0.05,
    device=config["general"]["device"],
)
print("Detector initialized.")

Detector initialized.


In [6]:
# List available raw photos
extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
raw_images = sorted(
    f for f in glob.glob(os.path.join(RAW_PHOTOS_DIR, "*"))
    if os.path.splitext(f)[1].lower() in extensions
)
print(f"Found {len(raw_images)} raw image(s) in {RAW_PHOTOS_DIR}")

if raw_images:
    for img_path in raw_images[:5]:
        print(f"  • {os.path.basename(img_path)}")

Found 0 raw image(s) in data/personal/raw


In [ ]:
# Run detection on a single image (if available)
if raw_images:
    demo_path = raw_images[0]
    result = detector.detect(demo_path)
    print(f"\nImage: {os.path.basename(demo_path)}")
    print(f"Image size: {result.image_width} x {result.image_height}")
    print(f"Persons detected: {result.num_persons}")
    for i, det in enumerate(result.detections):
        print(f"  Person {i+1}: bbox={det.bbox}, conf={det.confidence:.3f}, "
              f"size={det.width}x{det.height}, AR={det.aspect_ratio:.2f}")
else:
    print("No images found. Place raw photos in:", RAW_PHOTOS_DIR)

In [ ]:
# Visualize detections with bounding boxes and crops
if raw_images:
    detector.visualize_detections(demo_path, result=result)

### Detection Edge Cases

The pipeline handles several edge cases automatically:
- **Multiple persons per image**: Each person gets an individual crop.
- **Overlapping bounding boxes**: YOLO's built-in NMS eliminates duplicates (IoU threshold=0.45).
- **Minimum size filtering**: Crops below 25×50 px are discarded.
- **Aspect ratio**: Only keeps crops with H/W in [1.0, 6.0] — avoids horizontal slices or thin artifacts.
- **Confidence threshold**: Detections below the threshold (default 0.5) are dropped.
- **Padding**: 5% padding around each box captures partial context without background noise.

In [ ]:
# Visualize edge cases: show all images with multiple detections
if len(raw_images) > 1:
    multi_person_images = []
    for img_path in raw_images[:20]:
        res = detector.detect(img_path)
        if res.num_persons > 1:
            multi_person_images.append((img_path, res))

    if multi_person_images:
        print(f"Found {len(multi_person_images)} images with multiple persons:")
        for img_path, res in multi_person_images[:3]:
            detector.visualize_detections(img_path, result=res)
    else:
        print("No multi-person images found in the first 20 images.")

## 3. Batch Processing — Raw Photos → Crops

Process all raw photos and save detected person crops with metadata.

In [ ]:
if raw_images:
    summary = detector.process_directory(
        input_dir=RAW_PHOTOS_DIR,
        output_dir=CROPS_DIR,
        save_metadata=True,
    )
    print(f"\nTotal images processed: {summary['total_images']}")
    print(f"Total crops extracted : {summary['total_crops']}")
else:
    print("No raw images to process. Add photos to:", RAW_PHOTOS_DIR)

In [ ]:
# Display a grid of extracted crops
crop_files = sorted(glob.glob(os.path.join(CROPS_DIR, "*.jpg")))
n_show = min(len(crop_files), 16)

if n_show > 0:
    cols = min(n_show, 8)
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 4 * rows))
    axes = np.array(axes).flatten() if n_show > 1 else [axes]

    for i in range(n_show):
        img = Image.open(crop_files[i]).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(os.path.basename(crop_files[i]), fontsize=7)
        axes[i].axis("off")

    for i in range(n_show, len(axes)):
        axes[i].axis("off")

    plt.suptitle(f"Extracted Person Crops ({n_show}/{len(crop_files)})", fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No crops found yet.")

## 4. Manual Annotation Workflow

After crops are extracted, they need identity labels for supervised training.

### Annotation format
Create a `labels.json` file mapping each crop filename to an integer identity ID:

```json
{
    "photo1_crop000.jpg": 0,
    "photo1_crop001.jpg": 1,
    "photo2_crop000.jpg": 0,
    "photo3_crop000.jpg": 2,
    ...
}
```

### Steps:
1. Open the crops directory and visually inspect each crop.
2. Assign the same integer ID to crops of the same person.
3. Save the mapping as `labels.json` inside the crops directory.

The helper below generates a **template** file pre-populated with all crop filenames.

In [ ]:
def generate_annotation_template(crops_dir, output_path=None):
    """Generate a labels.json template with all crop filenames set to -1 (unlabeled)."""
    if output_path is None:
        output_path = os.path.join(crops_dir, "labels.json")

    crop_files = sorted(
        f for f in os.listdir(crops_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
        and f != "detection_metadata.json"
    )

    # Only create if doesn't exist (don't overwrite manual work)
    if os.path.exists(output_path):
        print(f"Labels file already exists: {output_path}")
        with open(output_path, "r") as f:
            existing = json.load(f)
        labeled = sum(1 for v in existing.values() if v >= 0)
        print(f"  {labeled}/{len(existing)} crops have been labeled.")
        return output_path

    template = {fname: -1 for fname in crop_files}
    with open(output_path, "w") as f:
        json.dump(template, f, indent=2)

    print(f"Annotation template created: {output_path}")
    print(f"  {len(template)} crops to label.")
    print("  Edit the file and replace -1 with identity IDs (0, 1, 2, ...).")
    return output_path


if crop_files:
    labels_path = generate_annotation_template(CROPS_DIR)
else:
    print("No crops available — run the batch processing step first.")

In [ ]:
# Visual annotation helper: display crops in a grid with filenames for easy labeling
def display_crops_for_labeling(crops_dir, start=0, count=20, cols=5):
    """Display crops in a numbered grid to facilitate manual annotation."""
    crop_files_local = sorted(
        f for f in os.listdir(crops_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
        and f != "detection_metadata.json"
    )
    subset = crop_files_local[start:start + count]
    if not subset:
        print("No crops to display.")
        return

    rows = (len(subset) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 5 * rows))
    axes = np.array(axes).flatten()

    for i, fname in enumerate(subset):
        img = Image.open(os.path.join(crops_dir, fname)).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(f"#{start + i}\n{fname}", fontsize=7)
        axes[i].axis("off")

    for i in range(len(subset), len(axes)):
        axes[i].axis("off")

    plt.suptitle(f"Crops {start}–{start + len(subset) - 1} (for annotation)", fontweight="bold")
    plt.tight_layout()
    plt.show()


if crop_files:
    display_crops_for_labeling(CROPS_DIR, start=0, count=20)

## 5. Dataset Splitting

Once labels are assigned, split the annotated crops into two subsets:

- **`personal_part1`**: Used for fine-tuning (domain adaptation).
- **`personal_part2`**: Held out for evaluation — never seen during training.

Two splitting strategies are supported:
- **Identity-disjoint**: Each identity appears in exactly one split. Best for measuring generalization.
- **Proportional**: Each identity is proportionally split across both parts. Useful when identity count is very small.

In [ ]:
from src.detection.yolo_pipeline import create_dataset_split

LABELS_PATH = os.path.join(CROPS_DIR, "labels.json")
SPLIT_OUTPUT = os.path.join(os.path.dirname(CROPS_DIR), "splits")

if os.path.exists(LABELS_PATH):
    # Check labeling completeness
    with open(LABELS_PATH, "r") as f:
        labels = json.load(f)
    labeled = {k: v for k, v in labels.items() if v >= 0}
    unlabeled = len(labels) - len(labeled)

    print(f"Total crops   : {len(labels)}")
    print(f"Labeled       : {len(labeled)}")
    print(f"Unlabeled (-1): {unlabeled}")
    print(f"Identities    : {len(set(labeled.values()))}")

    if unlabeled > 0:
        print(f"\n⚠ {unlabeled} crops are still unlabeled. "
              f"Complete labeling in {LABELS_PATH} before splitting.")
else:
    print(f"Labels file not found: {LABELS_PATH}")
    print("Run annotation template generation and label your crops first.")

In [ ]:
# Perform the split (only if labels are complete)
if os.path.exists(LABELS_PATH):
    with open(LABELS_PATH, "r") as f:
        labels = json.load(f)
    labeled = {k: v for k, v in labels.items() if v >= 0}

    if len(labeled) > 0 and all(v >= 0 for v in labeled.values()):
        # Save a clean labels file (only labeled entries) for the split function
        clean_labels_path = os.path.join(CROPS_DIR, "labels_clean.json")
        with open(clean_labels_path, "w") as f:
            json.dump(labeled, f, indent=2)

        split_stats = create_dataset_split(
            crops_dir=CROPS_DIR,
            labels_path=clean_labels_path,
            output_dir=SPLIT_OUTPUT,
            split_ratio=0.5,
            strategy="identity_disjoint",
            seed=config["general"]["seed"],
        )
    else:
        print("Not all crops are labeled yet. Complete annotation first.")
else:
    print("Skipping split — no labels file found.")

## 6. PersonalDataset Class Integration

Verify the `PersonalDataset` class works with the same API as `MarketDataset`.

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader

from src.dataloaders.personal_dataset import PersonalDataset

# Use the same transforms as Market-1501
IMG_SIZE = tuple(config["market1501"]["img_size"])  # (256, 128)

transform_test = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Try loading from crops directory (with or without labels)
PART1_DIR = os.path.join(SPLIT_OUTPUT, "personal_part1")
PART2_DIR = os.path.join(SPLIT_OUTPUT, "personal_part2")

# Load Part 1 (fine-tuning set)
if os.path.isdir(PART1_DIR):
    part1_labels = os.path.join(PART1_DIR, "labels.json")
    dataset_part1 = PersonalDataset(
        root_dir=PART1_DIR,
        labels_path=part1_labels if os.path.exists(part1_labels) else None,
        transform=transform_test,
    )
    print(f"Part 1: {dataset_part1.num_images} images, {dataset_part1.num_identities} identities")
else:
    print(f"Part 1 not found at {PART1_DIR}. Run the split step first.")
    dataset_part1 = None

# Load Part 2 (evaluation set)
if os.path.isdir(PART2_DIR):
    part2_labels = os.path.join(PART2_DIR, "labels.json")
    dataset_part2 = PersonalDataset(
        root_dir=PART2_DIR,
        labels_path=part2_labels if os.path.exists(part2_labels) else None,
        transform=transform_test,
    )
    print(f"Part 2: {dataset_part2.num_images} images, {dataset_part2.num_identities} identities")
else:
    print(f"Part 2 not found at {PART2_DIR}. Run the split step first.")
    dataset_part2 = None

In [ ]:
# Verify the dataset returns (image, label, camid) like MarketDataset
if dataset_part1 is not None and len(dataset_part1) > 0:
    img, label, camid = dataset_part1[0]
    print(f"Image shape : {img.shape}")
    print(f"Label       : {label}")
    print(f"Camera ID   : {camid}")

    # Test DataLoader compatibility
    loader = DataLoader(dataset_part1, batch_size=4, shuffle=False)
    batch_imgs, batch_labels, batch_camids = next(iter(loader))
    print(f"\nBatch shapes: images={batch_imgs.shape}, labels={batch_labels.shape}, camids={batch_camids.shape}")
    print("DataLoader integration OK.")
else:
    print("No data available to test. Complete the annotation and splitting steps.")

## 7. Exploratory Data Analysis

Analyze the personal dataset: identity distribution, image quality, crop sizes.

In [ ]:
def personal_dataset_eda(crops_dir, labels_path=None):
    """
    Comprehensive EDA for the personal dataset.
    Works with or without labels.
    """
    # --- Collect image statistics ---
    image_files = sorted(
        f for f in os.listdir(crops_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
        and not f.endswith(".json")
    )

    if not image_files:
        print("No crop images found for EDA.")
        return

    widths, heights, aspect_ratios, file_sizes = [], [], [], []
    for fname in image_files:
        fpath = os.path.join(crops_dir, fname)
        img = Image.open(fpath)
        w, h = img.size
        widths.append(w)
        heights.append(h)
        aspect_ratios.append(h / max(w, 1))
        file_sizes.append(os.path.getsize(fpath) / 1024)  # KB

    df_stats = pd.DataFrame({
        "filename": image_files,
        "width": widths,
        "height": heights,
        "aspect_ratio": aspect_ratios,
        "file_size_kb": file_sizes,
    })

    print(f"{'='*50}")
    print(f"Personal Dataset EDA — {len(image_files)} crops")
    print(f"{'='*50}")
    print(f"\nImage dimensions:")
    print(f"  Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
    print(f"  Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
    print(f"  Aspect ratio (H/W) — mean: {np.mean(aspect_ratios):.2f}, std: {np.std(aspect_ratios):.2f}")
    print(f"  File size — mean: {np.mean(file_sizes):.1f} KB, total: {sum(file_sizes)/1024:.1f} MB")

    # --- Plot 1: Dimension distributions ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(widths, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
    axes[0].set_title("Crop Width Distribution")
    axes[0].set_xlabel("Width (px)")
    axes[0].axvline(np.mean(widths), color="red", linestyle="--", label=f"Mean={np.mean(widths):.0f}")
    axes[0].legend()

    axes[1].hist(heights, bins=30, color="coral", edgecolor="white", alpha=0.8)
    axes[1].set_title("Crop Height Distribution")
    axes[1].set_xlabel("Height (px)")
    axes[1].axvline(np.mean(heights), color="red", linestyle="--", label=f"Mean={np.mean(heights):.0f}")
    axes[1].legend()

    axes[2].hist(aspect_ratios, bins=30, color="mediumseagreen", edgecolor="white", alpha=0.8)
    axes[2].set_title("Aspect Ratio (H/W) Distribution")
    axes[2].set_xlabel("H/W")
    axes[2].axvline(np.mean(aspect_ratios), color="red", linestyle="--", label=f"Mean={np.mean(aspect_ratios):.2f}")
    axes[2].legend()

    plt.suptitle("Crop Image Statistics", fontweight="bold")
    plt.tight_layout()
    plt.show()

    # --- Plot 2: Width vs Height scatter ---
    plt.figure(figsize=(6, 5))
    plt.scatter(widths, heights, alpha=0.5, s=15, color="steelblue")
    plt.xlabel("Width (px)")
    plt.ylabel("Height (px)")
    plt.title("Crop Dimensions: Width vs Height")
    plt.tight_layout()
    plt.show()

    # --- Identity distribution (if labels are available) ---
    if labels_path and os.path.exists(labels_path):
        with open(labels_path, "r") as f:
            labels = json.load(f)

        labeled = {k: v for k, v in labels.items() if v >= 0}
        if labeled:
            id_counts = Counter(labeled.values())
            n_ids = len(id_counts)
            counts = sorted(id_counts.values(), reverse=True)

            print(f"\nIdentity statistics:")
            print(f"  Unique identities: {n_ids}")
            print(f"  Images per identity — min: {min(counts)}, max: {max(counts)}, "
                  f"mean: {np.mean(counts):.1f}, median: {np.median(counts):.0f}")

            fig, axes = plt.subplots(1, 2, figsize=(14, 4))

            axes[0].bar(range(n_ids), counts, color="steelblue", edgecolor="white")
            axes[0].set_title("Images per Identity (sorted)")
            axes[0].set_xlabel("Identity rank")
            axes[0].set_ylabel("Number of images")
            axes[0].axhline(np.mean(counts), color="red", linestyle="--", label=f"Mean={np.mean(counts):.1f}")
            axes[0].legend()

            axes[1].hist(counts, bins=max(n_ids // 3, 5), color="coral", edgecolor="white", alpha=0.8)
            axes[1].set_title("Distribution of Images per Identity")
            axes[1].set_xlabel("Number of images")
            axes[1].set_ylabel("Number of identities")

            plt.suptitle("Identity Distribution Analysis", fontweight="bold")
            plt.tight_layout()
            plt.show()

    # --- Detection metadata (if available) ---
    meta_path = os.path.join(crops_dir, "detection_metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path, "r") as f:
            metadata = json.load(f)

        detections_per_image = [m["num_detections"] for m in metadata]
        confidences = [
            c["confidence"]
            for m in metadata
            for c in m.get("crops", [])
        ]

        print(f"\nDetection statistics:")
        print(f"  Source images: {len(metadata)}")
        print(f"  Persons per image — mean: {np.mean(detections_per_image):.1f}, "
              f"max: {max(detections_per_image)}")
        if confidences:
            print(f"  Detection confidence — mean: {np.mean(confidences):.3f}, "
                  f"min: {min(confidences):.3f}, max: {max(confidences):.3f}")

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].hist(detections_per_image, bins=range(max(detections_per_image) + 2),
                     color="mediumpurple", edgecolor="white", alpha=0.8, align="left")
        axes[0].set_title("Persons Detected per Source Image")
        axes[0].set_xlabel("Number of persons")
        axes[0].set_ylabel("Number of images")

        if confidences:
            axes[1].hist(confidences, bins=30, color="goldenrod", edgecolor="white", alpha=0.8)
            axes[1].set_title("Detection Confidence Distribution")
            axes[1].set_xlabel("Confidence")
            axes[1].set_ylabel("Count")

        plt.suptitle("YOLO Detection Analysis", fontweight="bold")
        plt.tight_layout()
        plt.show()

    return df_stats

In [ ]:
if os.path.isdir(CROPS_DIR) and os.listdir(CROPS_DIR):
    labels_file = os.path.join(CROPS_DIR, "labels.json")
    eda_stats = personal_dataset_eda(
        crops_dir=CROPS_DIR,
        labels_path=labels_file if os.path.exists(labels_file) else None,
    )
else:
    print("No crops available for EDA. Run detection pipeline first.")

### Comparison with Market-1501

Compare key statistics between the personal dataset and Market-1501.

In [ ]:
from src.dataloaders.market_dataset import MarketDataset

market_root = config["market1501"]["root_dir"]

if os.path.isdir(market_root):
    market_train = MarketDataset(market_root, subset="train")

    comparison = pd.DataFrame({
        "Metric": ["Total images", "Unique identities", "Images/identity (mean)"],
        "Market-1501 (train)": [
            len(market_train),
            len(market_train.pid_map),
            f"{len(market_train) / max(len(market_train.pid_map), 1):.1f}",
        ],
    })

    # Add personal dataset stats if available
    if dataset_part1 is not None:
        total_personal = (dataset_part1.num_images +
                          (dataset_part2.num_images if dataset_part2 else 0))
        total_ids = (dataset_part1.num_identities +
                     (dataset_part2.num_identities if dataset_part2 else 0))
        comparison["Personal Dataset"] = [
            total_personal,
            total_ids,
            f"{total_personal / max(total_ids, 1):.1f}",
        ]

    print(comparison.to_string(index=False))
else:
    print(f"Market-1501 not found at {market_root}. Download it first.")

---

## Summary

This notebook implements the complete **WP3 pipeline**:

| Step | Status | Output |
|------|--------|--------|
| 3.1 YOLO detection pipeline | `src/detection/yolo_pipeline.py` | Person crops with confidence scores |
| 3.2 Edge case handling | Built into `YOLOPersonDetector` | NMS, min size, aspect ratio filtering |
| 3.3 Photo collection & annotation | Annotation template + visual helper | `labels.json` |
| 3.4 Dataset split | `create_dataset_split()` | `personal_part1/` + `personal_part2/` |
| 3.5 PersonalDataset class | `src/dataloaders/personal_dataset.py` | Same API as `MarketDataset` |
| 3.6 EDA | `personal_dataset_eda()` | Distribution plots & statistics |

**Next steps:**
- WP6 (Domain Adaptation): Use `personal_part1` for fine-tuning and `personal_part2` for evaluation.